## 2.1 序列模型 – 理论计算题

给定字符序列 `"ababc"`，词汇表 V = {a, b, c}。采用一阶马尔可夫模型，使用拉普拉斯平滑（加1平滑）估计条件概率。

公式：
$$p(x_t | x_{t-1}) = \frac{\text{count}(x_{t-1}, x_t) + 1}{\text{count}(x_{t-1}) + |V|}$$

序列长度为5，共4次转移：
- a → b : 2次（位置1->2, 3->4）
- b → a : 1次（位置2->3）
- b → c : 1次（位置4->5）
- 其他转移为0次

各状态计数：count(a)=2，count(b)=2，count(c)=1

计算得：
1. p(a | b) = (1 + 1) / (2 + 3) = 2/5 = 0.4
2. p(c | b) = (1 + 1) / (2 + 3) = 2/5 = 0.4

答案：p(a|b)=0.4，p(c|b)=0.4。

In [1]:
# 2.2 文本预处理与滑动窗口
import re
from collections import Counter

def preprocess_text(text, n):
    """
    将文本小写，去除标点，分词，构建词汇表，生成滑动窗口特征和标签。
    返回：vocab (dict: word->id), (features, labels) 
          features是ID列表的列表，labels是ID列表（最后一个为None忽略）
    """
    # 1. 小写，保留字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 去除非字母和空格
    
    # 2. 分词
    words = text.split()
    if len(words) == 0:
        return {}, ([], [])
    
    # 3. 构建词汇表（按频率降序，分配ID从0开始）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.keys(), key=lambda w: word_counts[w], reverse=True)
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    ids = [vocab[w] for w in words]
    features = []
    labels = []
    for i in range(len(ids) - n):
        features.append(ids[i:i+n])
        labels.append(ids[i+n])
    return vocab, (features, labels)

# 测试
if __name__ == "__main__":
    text_example = "The time machine"
    vocab, (feat, lbl) = preprocess_text(text_example, n=2)
    print("词汇表:", vocab)
    print("特征 (ID):", feat)
    print("标签 (ID):", lbl)
    # 还原为词方便查看
    inv_vocab = {v: k for k, v in vocab.items()}
    print("特征 (词):", [[inv_vocab[id] for id in f] for f in feat])
    print("标签 (词):", [inv_vocab[id] for id in lbl])

词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征 (ID): [[0, 1]]
标签 (ID): [2]
特征 (词): [['the', 'time']]
标签 (词): ['machine']


## 3.1 RNN – 理论计算题

线性RNN（无偏置）：
- h_t = W_hh * h_{t-1} + W_hx * x_t
- o_t = W_oh * h_t
- 损失 L = 1/2 * Σ(o_t - y_t)^2

通过时间反向传播（BPTT）展开：

对于每个时间步 t，∂L/∂W_hh 是所有时间步贡献之和：
∂L/∂W_hh = Σ_{t=1}^T ∂L_t / ∂W_hh

利用链式法则，h_t 依赖于 h_{t-1}，因此递归展开：
∂h_t / ∂W_hh = h_{t-1}^T + W_hh * ∂h_{t-1} / ∂W_hh

最终梯度表达式（忽略具体形状转置）：
∂L/∂W_hh = Σ_{t=1}^T Σ_{k=1}^t [ (o_t - y_t) * W_oh * (Π_{j=k+1}^t W_hh) * ∂h_{k-1}/∂W_hh ]

梯度消失或爆炸的条件：
- 若 W_hh 的谱半径（最大特征值绝对值） < 1，连乘项指数衰减 → 梯度消失。
- 若谱半径 > 1，连乘项指数增长 → 梯度爆炸。
- 常用梯度裁剪或LSTM/GRU缓解。

In [2]:
# 3.2 RNN单元前向和反向传播（使用tanh）
import torch

def rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h):
    """
    前向传播: h_t = tanh(W_hh @ h_prev + W_xh @ x_t + b_h)
    输入形状：
        x_t: (batch_size, input_size)
        h_prev: (batch_size, hidden_size)
        W_hh: (hidden_size, hidden_size)
        W_xh: (hidden_size, input_size)
        b_h: (hidden_size,)
    返回 h_t (batch_size, hidden_size)
    """
    h_t = torch.tanh(torch.matmul(h_prev, W_hh.T) + torch.matmul(x_t, W_xh.T) + b_h)
    return h_t

def rnn_cell_backward(dh_next, x_t, h_prev, W_hh, W_xh, b_h):
    """
    反向传播：已知dh_next = dL/dh_t，计算dx_t, dh_prev, dW_hh, dW_xh, db_h
    """
    # 前向得到h_t
    h_t = torch.tanh(torch.matmul(h_prev, W_hh.T) + torch.matmul(x_t, W_xh.T) + b_h)
    # 导数 d tanh = 1 - tanh^2
    dtanh = 1 - h_t ** 2
    grad = dh_next * dtanh  # (batch_size, hidden_size)
    
    db_h = grad.sum(dim=0)
    dW_xh = torch.matmul(grad.T, x_t)
    dW_hh = torch.matmul(grad.T, h_prev)
    dx_t = torch.matmul(grad, W_xh)
    dh_prev = torch.matmul(grad, W_hh)
    
    return dx_t, dh_prev, dW_hh, dW_xh, db_h

# 简单测试
if __name__ == "__main__":
    batch_size, input_size, hidden_size = 2, 3, 4
    x_t = torch.randn(batch_size, input_size)
    h_prev = torch.randn(batch_size, hidden_size)
    W_hh = torch.randn(hidden_size, hidden_size)
    W_xh = torch.randn(hidden_size, input_size)
    b_h = torch.randn(hidden_size)
    
    h_t = rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h)
    print("h_t shape:", h_t.shape)
    
    dh_next = torch.ones_like(h_t)
    dx, dh, dWhh, dWxh, db = rnn_cell_backward(dh_next, x_t, h_prev, W_hh, W_xh, b_h)
    print("dx shape:", dx.shape)
    print("dh_prev shape:", dh.shape)
    print("dW_hh shape:", dWhh.shape)
    print("dW_xh shape:", dWxh.shape)
    print("db shape:", db.shape)

h_t shape: torch.Size([2, 4])
dx shape: torch.Size([2, 3])
dh_prev shape: torch.Size([2, 4])
dW_hh shape: torch.Size([4, 4])
dW_xh shape: torch.Size([4, 3])
db shape: torch.Size([4])


## 4.1 高级RNN – 理论计算题

深度双向 RNN：L 层，每层隐藏单元数 H，输入维度 D。

每层双向包含前向和后向两个单向 RNN。每个单向 RNN 参数包括：
- 输入到隐藏权重 W_ih
- 隐藏到隐藏权重 W_hh
- 偏置 b_h

**第一层**（输入维度 D）：
每个方向参数：H*D (W_ih) + H*H (W_hh) + H (bias)
两个方向：2 * (HD + H^2 + H) = 2H(D + H + 1)

**第 2 至第 L 层**（输入来自前一层的拼接，维度为 2H）：
每个方向参数：H*(2H) (W_ih) + H*H (W_hh) + H (bias) = 3H^2 + H
两个方向：2*(3H^2 + H) = 6H^2 + 2H

**总参数量（忽略输出层）**：
Total = 2H(D + H + 1) + (L - 1)(6H^2 + 2H)

In [3]:
# 4.2 双向RNN编码器
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super(BiRNNEncoder, self).__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=False, 
                          bidirectional=True, dropout=dropout)
        self.hidden_dim = hidden_dim
        
    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        返回：
            outputs: (seq_len, batch, 2*hidden_dim) 每个时间步拼接
            final_hidden: (batch, 2*hidden_dim) 最后时间步的拼接（前向最后+后向最后）
        """
        outputs, h_n = self.rnn(X)  # h_n: (num_layers*2, batch, hidden_dim)
        # 取最后一层的前向和后向
        last_layer_forward = h_n[-2]   # (batch, hidden_dim)
        last_layer_backward = h_n[-1]  # (batch, hidden_dim)
        final_hidden = torch.cat([last_layer_forward, last_layer_backward], dim=1)
        return outputs, final_hidden

# 测试
if __name__ == "__main__":
    seq_len, batch, input_dim = 5, 3, 8
    hidden_dim = 4
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    X = torch.randn(seq_len, batch, input_dim)
    out, final = encoder(X)
    print("输出形状 (seq_len, batch, 2*H):", out.shape)
    print("最终隐藏状态 (batch, 2*H):", final.shape)

输出形状 (seq_len, batch, 2*H): torch.Size([5, 3, 8])
最终隐藏状态 (batch, 2*H): torch.Size([3, 8])


## 5.1 嵌入向量 – 理论计算题

Skip-gram 负采样损失函数（中心词 w_c，上下文词 w_o，采样 K 个负样本 n_k）：

目标是最小化负对数似然：
L = -log σ(u_wo^T * v_wc) - Σ_{k=1}^K log σ(-u_nk^T * v_wc)

其中 σ(x) = 1 / (1 + e^{-x})。

负样本从噪声分布 P_n(w) 中采样，常用 unigram 分布的 3/4 次方（提高低频词采样概率）。

完整目标函数（对所有训练样本求和）：
J = - Σ_{(wc, wo)} [ log σ(u_wo^T v_wc) + Σ_{k=1}^K E_{nk ~ P_n} log σ(-u_nk^T v_wc) ]

In [4]:
# 5.2 CBOW 完整 softmax 前向和损失
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_idx, W, W_out):
    """
    context_indices: (batch_size, context_size) 上下文词索引
    target_idx: (batch_size,) 目标中心词索引
    W: (V, d) 输入权重矩阵
    W_out: (d, V) 输出权重矩阵
    返回：交叉熵损失（标量）
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    # 取每个上下文词的嵌入 (batch, context_size, d)
    emb = W[context_indices]  # (batch, context_size, d)
    # 平均上下文向量 (batch, d)
    h = emb.mean(dim=1)  # (batch, d)
    # 计算输出得分 (batch, V)
    scores = torch.matmul(h, W_out)  # (batch, V)
    # 计算交叉熵损失
    loss = F.cross_entropy(scores, target_idx)
    return loss

# 测试
if __name__ == "__main__":
    V, d = 10, 4
    context_size = 2
    batch = 3
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    context_indices = torch.randint(0, V, (batch, context_size))
    target_idx = torch.randint(0, V, (batch,))
    loss = cbow_forward(context_indices, target_idx, W, W_out)
    print("CBOW损失:", loss.item())

CBOW损失: 2.363767385482788


## 6.1 注意力机制 – 理论计算题

给定 Q ∈ R^(2×4), K ∈ R^(3×4), V ∈ R^(3×5)，d_k = 4。

缩放点积注意力步骤：

1. **得分矩阵** S = Q * K^T / sqrt(d_k)，形状 2×3。
   设 Q = [q1; q2], K = [k1; k2; k3]，则 S_ij = (qi · kj) / 2。

2. **Softmax**：对每一行（每个查询）做 softmax，得到注意力权重 A，形状 2×3。
   A_ij = exp(S_ij) / Σ_j' exp(S_ij')

3. **加权求和**：输出 O = A * V，形状 2×5。
   O_i = Σ_j A_ij * v_j

结果矩阵 O ∈ R^(2×5)。

In [5]:
# 6.2 多头注意力前向传播
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        # 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 重塑为 (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        
        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)  # (batch, num_heads, seq_len, d_k)
        
        # 拼接并恢复形状
        out = out.permute(2, 0, 1, 3).contiguous().view(seq_len, batch, self.d_model)
        out = self.W_o(out)
        return out

# 测试
if __name__ == "__main__":
    d_model = 4
    num_heads = 2
    seq_len, batch = 3, 2
    X = torch.randn(seq_len, batch, d_model)
    mha = MultiHeadAttention(d_model, num_heads)
    out = mha(X)
    print("输入形状:", X.shape)
    print("输出形状:", out.shape)  # 应与输入相同

输入形状: torch.Size([3, 2, 4])
输出形状: torch.Size([3, 2, 4])
